In [11]:
# 🔷 Question 1: Build an End-to-End RAG + Agent System (25 Marks)
# 🧩 Scenario
# You are an AI intern at an ed-tech company. Students frequently ask questions about:

# Course policies (refunds, deadlines)
# Lecture content (PDF notes)
# Assignment deadlines
# Their enrollment status
# The company wants a single intelligent assistant that:

# Answers questions using internal documents (PDFs, FAQs)
# Fetches student-specific data (like enrollment or progress) using tools/APIs
# Avoids hallucination and gives reliable answers
# 💻 Task
# Design and implement a working prototype (pseudo-code or real code) for this system.

# Your solution must include:

# ✅ 1. RAG Pipeline
# How you will ingest and preprocess documents
# Chunking strategy (with justification)
# Embedding + vector store choice
# Retrieval logic
# How context is injected into the LLM
# ✅ 2. Agent Integration
# Design an agent that decides:
# When to use RAG
# When to call a tool (e.g., get_student_status(student_id))
# Show how tools are defined and used
# ✅ 3. End-to-End Flow
# Write code or structured pseudo-code showing:

# Input query
# Retrieval step
# Tool calling (if needed)
# Final answer generation
# ✅ 4. Reliability Improvements
# Show at least 2 techniques in code/design to:

# Reduce hallucination
# Improve answer quality
# 🎯 Expected Outcome
# A working architecture/code that demonstrates:

# RAG + Agent working together
# Decision-making capability
# Real-world practicality



In [ ]:

# IMPORT LIBRARIES

import os
import faiss
import numpy as np
from groq import Groq
from sentence_transformers import SentenceTransformer


# SET GROQ API KEY

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# STEP 1: INTERNAL DOCUMENT DATA

documents = [
    "Refund policy: Students can request a refund within 7 days of enrollment.",
    "Assignment deadline: Assignment 1 is due on March 30.",
    "Lecture: Neural Networks consist of layers, weights, and activation functions.",
    "Course policy: Late submissions incur a 10 percent penalty per day.",
    "Lecture: RAG combines retrieval and generation to improve accuracy."
]


# STEP 2: CHUNKING

def chunk_text(text, chunk_size=40):

    words = text.split()

    chunks = [
        " ".join(words[i:i+chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

    return chunks


chunks = []

for doc in documents:
    chunks.extend(chunk_text(doc))


# STEP 3: EMBEDDINGS

embed_model = SentenceTransformer("paraphrase-MiniLM-L3-v2")
embeddings = embed_model.encode(chunks)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))


# STEP 4: RETRIEVAL

def retrieve(query, k=2):

    query_embedding = embed_model.encode([query])

    distances, indices = index.search(np.array(query_embedding), k)

    results = []

    for i, d in zip(indices[0], distances[0]):

        if d < 1.5:
            results.append(chunks[i])

    return results


# STEP 5: LLM WITH GROQ

def generate_answer(query, context):

    if not context:
        return "I don't know based on the course documents."

    prompt = f"""
You are an AI assistant for an ed-tech platform.

Answer ONLY using the provided context.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(

        model="llama3-8b-8192",

        messages=[
            {"role": "user", "content": prompt}
        ]

    )

    return response.choices[0].message.content

# STEP 6: TOOL (STUDENT DATABASE)

student_db = {

    "101": {"status": "enrolled", "progress": "60%"},
    "102": {"status": "not enrolled", "progress": "0%"},
    "103": {"status": "enrolled", "progress": "85%"}
}


def get_student_status(student_id):

    return student_db.get(student_id, "Student not found")

# STEP 7: AGENT DECISION

def agent_decision(query):

    query = query.lower()

    if "my" in query or "status" in query or "progress" in query:
        return "tool"

    return "rag"

# STEP 8: END TO END SYSTEM

def system(query, student_id=None):

    decision = agent_decision(query)

    print("\n[Agent Decision]:", decision.upper())

    if decision == "tool":

        result = get_student_status(student_id)

        return f"Tool Response → {result}"

    if decision == "rag":

        context = retrieve(query)

        print("[Retrieved Context]:", context)

        answer = generate_answer(query, context)

        return answer



# STEP 9: TEST CASES

print("---- TEST 1: RAG QUERY ----")
print(system("What is the refund policy?"))

print("\n---- TEST 2: DEADLINE QUERY ----")
print(system("When is assignment due?"))

print("\n---- TEST 3: TOOL QUERY ----")
print(system("What is my enrollment status?", student_id="101"))

print("\n---- TEST 4: UNKNOWN QUERY ----")
print(system("Explain quantum physics"))

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L3-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


---- TEST 1: RAG QUERY ----

[Agent Decision]: RAG
[Retrieved Context]: []
I don't know based on the course documents.

---- TEST 2: DEADLINE QUERY ----

[Agent Decision]: RAG
[Retrieved Context]: []
I don't know based on the course documents.

---- TEST 3: TOOL QUERY ----

[Agent Decision]: TOOL
Tool Response → {'status': 'enrolled', 'progress': '60%'}

---- TEST 4: UNKNOWN QUERY ----

[Agent Decision]: RAG
[Retrieved Context]: []
I don't know based on the course documents.
